# 07 - SQL Analysis

Runs the scripts in `sql/` against a local SQLite database and shows the results. The same queries work in MySQL or PostgreSQL with minor syntax changes (`CAST`, `ROUND`, `||`).

In [1]:
import sqlite3
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

raw = pd.read_excel("../Data/Raw/Telco_customer_churn.xlsx")
raw.columns = [c.strip().lower().replace(" ", "_") for c in raw.columns]
raw = raw.rename(columns={"customerid": "customer_id"})
raw["total_charges"] = raw["total_charges"].astype(str)

con = sqlite3.connect(":memory:")
con.executescript(open("../sql/01_create_tables.sql").read())
raw.to_sql("customers_raw", con, if_exists="append", index=False)
con.executescript(open("../sql/02_data_cleaning.sql").read())
pd.read_sql("SELECT COUNT(*) AS rows_loaded FROM customers", con)

,rows_loaded
0,7043


In [2]:
def run_file(path):
    text = open(path).read()
    results = []
    for block in text.split(";"):
        lines = [l for l in block.strip().splitlines()]
        if not any(l.strip().upper().startswith(("SELECT", "WITH")) for l in lines):
            continue
        title = next((l.strip("- ").strip() for l in lines if l.startswith("--")), "")
        results.append((title, pd.read_sql(block, con)))
    return results

analysis = run_file("../sql/03_churn_analysis.sql")
len(analysis)

10

In [3]:
for title, result in analysis[:5]:
    print(title)
    print(result.to_string(index=False))
    print()

Churn analysis queries (SQLite syntax)
 customers  churned  churn_rate_pct
      7043     1869           26.54

2. Churn by contract type
      contract  customers  churned  churn_rate_pct
Month-to-month       3875     1655           42.71
      One year       1473      166           11.27
      Two year       1695       48            2.83

3. Churn by tenure group
tenure_group  customers  churned  churn_rate_pct
 0-12 months       2186     1037           47.44
13-24 months       1024      294           28.71
25-48 months       1594      325           20.39
  49+ months       2239      213            9.51

4. Churn by internet service
internet_service  customers  churn_rate_pct  avg_monthly_charges
     Fiber optic       3096           41.89                91.50
             DSL       2421           18.96                58.10
              No       1526            7.40                21.08

5. Churn by payment method
           payment_method  customers  churn_rate_pct
         Electro

In [4]:
for title, result in analysis[5:]:
    print(title)
    print(result.to_string(index=False))
    print()

6. Churn by demographics
       segment  value  customers  churn_rate_pct
Senior Citizen     No       5901           23.61
Senior Citizen    Yes       1142           41.68
       Partner     No       3641           32.96
       Partner    Yes       3402           19.66
    Dependents     No       5416           32.55
    Dependents    Yes       1627            6.52
        Gender Female       3488           26.92
        Gender   Male       3555           26.16

7. Add-on services and churn (internet customers only)
          service has_service  customers  churn_rate_pct
  Online Security          No       3498           41.77
  Online Security         Yes       2019           14.61
     Tech Support          No       3473           41.64
     Tech Support         Yes       2044           15.17
    Online Backup          No       3088           39.93
    Online Backup         Yes       2429           21.53
Device Protection          No       3095           39.13
Device Protection     

## Business KPIs

In [5]:
kpis = run_file("../sql/04_business_kpis.sql")
for title, result in kpis[:6]:
    print(title)
    print(result.to_string(index=False))
    print()

Revenue and value KPIs (SQLite syntax)
 total_mrr  mrr_lost  mrr_lost_pct
  456116.6 139130.85          30.5

2. Average customer value: churned vs retained
churn_label  customers  avg_monthly_charges  avg_total_charges  avg_tenure  avg_cltv
         No       5174                61.27            2549.91        37.6    4491.0
        Yes       1869                74.44            1531.80        18.0    4149.0

3. Lifetime value lost by contract
      contract  churned  cltv_lost  mrr_lost
Month-to-month     1655  6757185.0 120847.10
      One year      166   750522.0  14118.45
      Two year       48   247549.0   4165.30

4. Share of churned customers that were month-to-month
 pct_of_churn_from_month_to_month
                            88.55

5. Monthly charge bands
charge_band  customers  churn_rate_pct  mrr_lost
   Under 35       1731           10.86   4423.05
      35-70       1721           23.65  21726.80
      70-90       1847           37.95  55926.90
        90+       1744     

## Check: SQL and pandas agree

Quick reconciliation between the SQL output and the pandas cleaning step.

In [6]:
sql_df = pd.read_sql("SELECT customer_id, total_charges, churn_value FROM customers ORDER BY customer_id", con)
clean = pd.read_csv("../Data/Processed/telco_churn_clean.csv").sort_values("customer_id")

print("Row counts match      :", len(sql_df) == len(clean))
print("Total charges match   :", abs(sql_df["total_charges"].sum() - clean["total_charges"].sum()) < 0.01)
print("Churn counts match    :", sql_df["churn_value"].sum() == clean["churn_value"].sum())

Row counts match      : True
Total charges match   : True
Churn counts match    : True
